# 🎙️ Urdu Interview Processing Pipeline (CPU)
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used (CPU-optimized):**
- ASR: **faster-whisper** + **`int8` quantization** → `openai/whisper-large-v3-turbo`
- Translation: `facebook/nllb-200-distilled-600M` (~300 MB)
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Why faster-whisper + int8 on CPU?**
- Runs efficiently on a normal PC without GPU
- int8 keeps accuracy close to full precision while using less RAM
- Recommended CPU ASR setup for this pipeline

**CPU notes:**
- First run downloads models (~1.5 GB total); transcription is slower than GPU
- Works locally on Windows, macOS, Linux, or Colab CPU runtime

In [1]:
# ── CELL 1: Check runtime (CPU mode) ───────────────────────
import torch

print('CPU mode enabled — GPU will not be used.')
print(f'  PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print('  Note: a GPU is visible but this notebook forces CPU execution.')
else:
    print('  No GPU detected (expected for CPU setup).')

CPU mode enabled — GPU will not be used.
  PyTorch: 2.11.0+cpu
  No GPU detected (expected for CPU setup).


In [22]:
# ── CELL 2: Install all dependencies ──────────────────────
import sys

py = sys.version_info
print(f'Python {sys.version.split()[0]}')
if py >= (3, 13):
    print('Using Python 3.13+ compatible package pins (Windows-safe wheels).')
elif py < (3, 9):
    raise RuntimeError('Python 3.9+ is required.')

print('Installing dependencies... (takes 3-5 minutes first time)')
print('CPU ASR: faster-whisper + int8 (~800MB) | Translation: NLLB-600M (~1.2GB)')

# Pin versions that have prebuilt wheels on Python 3.13 / Windows
!pip install -q faster-whisper==1.1.0
!pip install -q "transformers==4.44.2" "sentencepiece>=0.2.2" sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.13
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model (compatible with spacy 3.8.x)
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ faster-whisper will use int8 quantization on CPU for ASR')

Python 3.13.2
Using Python 3.13+ compatible package pins (Windows-safe wheels).
Installing dependencies... (takes 3-5 minutes first time)
CPU ASR: faster-whisper + int8 (~800MB) | Translation: NLLB-600M (~1.2GB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [88 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp313-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\msaad\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\msaad\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\msaad\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: i

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')

✔ All dependencies installed!
✔ faster-whisper will use int8 quantization on CPU for ASR



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# ── CELL 3: Locate project (local or Colab clone) ──────────
import os

if os.path.isdir('pipeline') and os.path.isfile('config.py'):
    print('✔ Using existing project in current directory')
elif os.path.isdir('../pipeline') and os.path.isfile('../config.py'):
    os.chdir('..')
    print('✔ Using existing project in parent directory')
else:
    !git clone https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE-FOR-CPU-.git urdu-pipeline
    os.chdir('urdu-pipeline')
    !git pull origin main
    print('✔ Repository cloned from GitHub')

print('Project root:', os.getcwd())
print('Contents:', os.listdir('.'))

✔ Using existing project in current directory
Project root: c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2
Contents: ['.git', '.gitignore', 'align.py', 'audio', 'config.py', 'evaluate.py', 'main.py', 'metrics.py', 'normalize.py', 'notebooks', 'outputs', 'pipeline', 'README.md', 'requirements.txt', 'scripts', '__pycache__']


In [3]:
# ── CELL 4: Download audio from YouTube ───────────────────
import json
import os
import subprocess

os.makedirs('audio', exist_ok=True)

# ffmpeg/ffprobe required for trim + mp3 conversion (missing on many Windows installs)
try:
    subprocess.run(['ffmpeg', '-version'], capture_output=True, check=True)
except (FileNotFoundError, subprocess.CalledProcessError):
    !pip install -q static-ffmpeg
    import static_ffmpeg
    static_ffmpeg.add_paths()

!pip install -q yt-dlp

YOUTUBE_URL = "https://youtu.be/pHZHYWe8Mkc"

# ── EDIT THESE TWO LINES ONLY ─────────────────────────────
# Use MM:SS (e.g. "2:17") or plain seconds (e.g. 137).
START_AT = "2:17"   # start from here
END_AT   = "7:17"   # end here
# ─────────────────────────────────────────────────────────

def parse_time(value):
    """Accept seconds (137) or timestamps like 2:17 or 1:02:30."""
    if isinstance(value, (int, float)):
        return int(value)
    text = str(value).strip()
    if text.isdigit():
        return int(text)
    parts = [int(p) for p in text.split(':')]
    if len(parts) == 2:
        minutes, seconds = parts
        return minutes * 60 + seconds
    if len(parts) == 3:
        hours, minutes, seconds = parts
        return hours * 3600 + minutes * 60 + seconds
    raise ValueError(f"Invalid time format: {value!r}. Use seconds or MM:SS / HH:MM:SS.")

def format_time(seconds):
    seconds = int(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"

start_seconds = parse_time(START_AT)
end_seconds = parse_time(END_AT)
if end_seconds <= start_seconds:
    raise ValueError(f"END_AT ({format_time(end_seconds)}) must be after START_AT ({format_time(start_seconds)}).")

clip_seconds = end_seconds - start_seconds

# Download full audio as mp3
!yt-dlp -x --audio-format mp3 -o "audio/full.%(ext)s" "{YOUTUBE_URL}"

full_audio = "audio/full.mp3"
if not os.path.exists(full_audio):
    for ext in ("webm", "m4a", "opus", "wav"):
        candidate = f"audio/full.{ext}"
        if os.path.exists(candidate):
            full_audio = candidate
            break
    else:
        raise FileNotFoundError("Download finished but no audio/full.* file was found.")

trim = subprocess.run(
    [
        "ffmpeg", "-y", "-i", full_audio,
        "-ss", str(start_seconds),
        "-t", str(clip_seconds),
        "-c", "copy",
        "audio/test_audio.mp3",
    ],
    capture_output=True,
    text=True,
)
if trim.returncode != 0:
    raise RuntimeError(f"ffmpeg trim failed:\n{trim.stderr}")

audio_path = "audio/test_audio.mp3"
size_mb = os.path.getsize(audio_path) / 1e6
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", audio_path],
    capture_output=True,
    text=True,
    check=True,
)
duration_min = float(json.loads(probe.stdout)["format"]["duration"]) / 60

print(f"\n✔ Audio ready: {audio_path}")
print(f"   Clip: {format_time(start_seconds)} → {format_time(end_seconds)} ({clip_seconds / 60:.1f} min requested)")
print(f"   File size: {size_mb:.1f} MB")
print(f"   Actual duration: {duration_min:.1f} minutes")



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


[youtube] Extracting URL: https://youtu.be/pHZHYWe8Mkc
[youtube] pHZHYWe8Mkc: Downloading webpage
[youtube] pHZHYWe8Mkc: Downloading android vr player API JSON
[info] pHZHYWe8Mkc: Downloading 1 format(s): 251
[download] audio\full.mp3 has already been downloaded
[ExtractAudio] Not converting audio audio\full.mp3; file is already in target format mp3

✔ Audio ready: audio/test_audio.mp3
   Clip: 2:17 → 7:17 (5.0 min requested)
   File size: 2.9 MB
   Actual duration: 5.0 minutes


In [4]:
# ── CELL 5: Configure pipeline (CPU) ───────────────────────
import sys
import os

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import importlib
import config
importlib.reload(config)

# CPU ASR: faster-whisper + int8 quantization
config.WHISPER_DEVICE       = 'cpu'
config.WHISPER_COMPUTE_TYPE = 'int8'

# Force Urdu-only transcription (do not use auto-detect for this pipeline)
config.WHISPER_LANGUAGE = 'ur'
config.WHISPER_INITIAL_PROMPT = (
    'یہ ایک اردو زبان کا انٹرویو ہے۔ مکمل گفتگو اردو رسم الخط میں لکھی جائے۔'
)

AUDIO_PATH = audio_path

print('Pipeline Configuration (CPU):')
print(f'  ASR              : faster-whisper / whisper-{config.WHISPER_MODEL}')
print(f'  ASR quantization : {config.WHISPER_COMPUTE_TYPE}')
print(f'  ASR language     : {config.WHISPER_LANGUAGE} (Urdu only)')
print(f'  Translation Model: {config.CPU_TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration (CPU):
  ASR              : faster-whisper / whisper-large-v3-turbo
  ASR quantization : int8
  ASR language     : ur (Urdu only)
  Translation Model: facebook/nllb-200-distilled-600M
  Device           : cpu
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.55
  Chunk size       : 500 chars


In [6]:
# Check if audio file exists and confirm CPU mode
import json
import os
import subprocess

def _ensure_ffprobe():
    try:
        subprocess.run(['ffprobe', '-version'], capture_output=True, check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        import static_ffmpeg
        static_ffmpeg.add_paths()

audio_path = "audio/test_audio.mp3"
duration_min = None
if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / 1e6
    _ensure_ffprobe()
    probe = subprocess.run(
        ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', audio_path],
        capture_output=True,
        text=True,
        check=True,
    )
    duration_min = float(json.loads(probe.stdout)['format']['duration']) / 60
    print(f'✔ Audio file exists: {size_mb:.1f} MB ({duration_min:.1f} minutes)')
else:
    print(f'✗ Audio file NOT found: {audio_path}')
    print('  Run Cell 4 first to download and trim the audio.')

print(f'✔ Device: {config.WHISPER_DEVICE} ({config.WHISPER_COMPUTE_TYPE})')
print(f'✔ Translation: {config.CPU_TRANSLATION_MODEL}')
if duration_min is not None:
    if duration_min > 10:
        print('  Tip: long audio on CPU can take 30+ minutes for transcription.')
    else:
        print('  Tip: short clips on CPU usually finish transcription in a few minutes.')

✔ Audio file exists: 2.9 MB (5.0 minutes)
✔ Device: cpu (int8)
✔ Translation: facebook/nllb-200-distilled-600M
  Tip: short clips on CPU usually finish transcription in a few minutes.


In [7]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: TRANSCRIPTION (ASR)
  Audio file   : audio/test_audio.mp3
  Model        : whisper-large-v3-turbo
  Backend      : faster-whisper
  Language     : ur
  Device       : cpu
  Compute type : int8
  Temperature  : 0.0 (fallback: [0.2, 0.4, 0.6])
  Beam size    : 5


c:\Users\msaad\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Loading faster-whisper model (int8 on CPU, ~800 MB first download)...
  Model loaded (cpu, int8).

  Transcribing audio (language: ur) ...
  Raw segments from Whisper: 54
  Merged 3 micro-segments into neighbours.

  Results:
  Segments total    : 51
    Urdu            : 51
    English         : 0
    Low confidence  : 0  (threshold=0.55)
  Avg raw confidence: 0.807
  Avg cal confidence: 0.887
  Avg text quality  : 0.814
  Loops removed     : 0
  Micro-segs merged : 3

  Segment detail:
    [ ][UR]    00:00:00.180 -> 00:00:06.440 | conf=0.928 | السلام علیکم Dr. Majda. Welcome to the show. Thank you 
    [ ][UR]    00:00:08.000 -> 00:00:17.180 | conf=0.845 | If you give us a small introduction so that our audienc
    [ ][UR]    00:00:17.380 -> 00:00:26.220 | conf=0.945 | Thank you so much Zainab for inviting me on your show. 
    [ ][UR]    00:00:32.880 -> 00:00:41.220 | conf=0.865 | بیٹے کے ساتھ ایسا ہو چکا ہے کہ اس نے کہا کہ میں مادر اس
    [ ][UR]    00:00:41.220 -> 00:00:44.040 

In [9]:
# Clear memory between heavy stages (CPU RAM)
import gc

for name in ('model', 'model_', 'tokenizer'):
    if name in globals():
        del globals()[name]
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass

In [10]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF TRANSCRIPT
  Interview ID        : test_audio
  Total segments      : 51
  Urdu segments       : 51
  English segments    : 0
  Micro-segs merged   : 3
  Loops removed       : 0
  Avg raw confidence  : 0.8072
  Avg cal confidence  : 0.8872
  Avg text quality    : 0.8141
  Confidence threshold: 0.55
    [ok      ][UR]    seg 001 | conf=0.928 | tq=0.94 | السلام علیکم Dr. Majda. Welcome to the show. 
    [ok      ][UR]    seg 002 | conf=0.845 | tq=0.99 | If you give us a small introduction so that o
    [ok      ][UR]    seg 003 | conf=0.945 | tq=0.97 | Thank you so much Zainab for inviting me on y
    [ok      ][UR]    seg 004 | conf=0.865 | tq=0.73 | بیٹے کے ساتھ ایسا ہو چکا ہے کہ اس نے کہا کہ م
    [ok      ][UR]    seg 005 | conf=0.637 | tq=0.76 | تو وہ کہتا ہے کہ وہ پہلی دکٹر ہے
    [ok      ][UR]    seg 006 | conf=0.787 | tq=0.70 | تو پہلی میں نے کیا ہے
    [ok      ][UR]    seg 007 | conf=0.866 | tq=0.91 | میرا ایریا اسپیشلیزیشن ڈیجیٹل سسٹم دیزائن ہے
   

In [23]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH (smart routing)
  Interview ID   : test_audio
  Model          : facebook/nllb-200-distilled-600M
  Routing        : Urdu→NLLB  |  English→pass-through

  Routing: 51 Urdu segments -> translate  |  0 English segments -> pass-through

  Loading translation model (~1.2GB first run)...


c:\Users\msaad\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 14182.20it/s]
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Model loaded on cpu.

  Translating Urdu segments...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


  Assembling translated segments:
    [T] seg 001/51 | TRANSLATED | Well- ScG. D. D. D. D.A. D. D.A. D. D. D. D. D.A. D. D. D. D...
    [T] seg 002/51 | TRANSLATED | I'll get back to the back-on-Splick-Splicks-Henaptaptier, Wy...
    [T] seg 003/51 | TRANSLATED | Despite this, I was able to get back to the next day, and I ...
    [T] seg 004/51 | TRANSLATED | He's like that with the son he said that I was the mother, s...
    [T] seg 005/51 | TRANSLATED | So he says, 'This is the first time';...
    [T] seg 006/51 | TRANSLATED | So the first thing I did....
    [T] seg 007/51 | TRANSLATED | My call is digital digital system system....
    [T] seg 008/51 | TRANSLATED | And I'm a professor of this exchange....
    [T] seg 009/51 | TRANSLATED | In the computer at the University of Pypans....
    [T] seg 010/51 | TRANSLATED | In addition to the University of Oila....
    [T] seg 011/51 | TRANSLATED | There's an army center....
    [T] seg 012/51 | TRANSLATED | which is now a digital syste

In [12]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID      : test_audio
  Total segments    : 51
  Translated        : 51
  Pass-through      : 0

  Checks: empty, error markers, length ratio, nonsense detection
    [✔][T] seg 001
             EN: Well- ScG. D. D. D. D.A. D. D.A. D. D. D. D. D.A. D. D. D. D. D. D. D....
    [✔][T] seg 002
             EN: I'll get back to the back-on-Splick-Splicks-Henaptaptier, Wynapt-wickl...
    [✔][T] seg 003
             EN: Despite this, I was able to get back to the next day, and I was able t...
    [✔][T] seg 004
             EN: He's like that with the son he said that I was the mother, so he said ...
    [✔][T] seg 005
             EN: So he says, 'This is the first time';...
    [✔][T] seg 006
             EN: So the first thing I did....
    [✔][T] seg 007
             EN: My call is digital digital system system....
    [✔][T] seg 008
             EN: And I'm a professor of this exchange....
    [✔][T] seg 009
             EN: In 

In [13]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...
  ✔ Presidio loaded.

  De-identifying full English text...

  De-identifying segments:
    [PII] seg 001 | 4 entities removed | Well- ScG. [ORGANIZATION] D. D.A. [NAME]...
    [PII] seg 002 | 3 entities removed | I'll get back to the back-on-Splick-Splicks-Henaptaptier, [N...
    [PII] seg 003 | 1 entities removed | Despite this, I was able to get back to [DATE], and I was ab...
    [ok ] seg 004 | 0 entities removed | He's like that with the son he said that I was the mother, s...
    [ok ] seg 005 | 0 entities removed | So he says, 'This is the first time';...
    [ok ] seg 006 | 0 entities removed | So the first thing I did....
    [ok ] seg 007 | 0 entities removed | My call is digital digital system system....
    [ok ] seg 008 | 0 entities removed | And I'm a professor of this exchange....
    [ok ] seg 009 | 0 entities removed | In the computer at the

In [14]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')


  STAGE 6: FINAL EXPORT: JSON + DOCX
  Interview ID : test_audio

  Building final JSON dataset...
  Saved -> c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2\outputs\6_final_dataset\test_audio_final_dataset.json

  Building DOCX report...
  ✔ DOCX saved → c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2\outputs\6_final_dataset\test_audio_final_dataset.docx

  ── Final Outputs ─────────────────────────────
  JSON → c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2\outputs\6_final_dataset\test_audio_final_dataset.json
  DOCX → c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2\outputs\6_final_dataset\test_audio_final_dataset.docx

  ✔ Stage 6 complete. Pipeline finished!

✔ Final JSON : c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2\outputs\6_final_dataset\test_audio_final_dataset.json
✔ Final DOCX : c:\Users\msaad\Downloads\urdu-pipeline2\urdu-pipeline2\outputs\6_final_dataset\test_audio_final_dataset.docx


In [15]:
# ── CELL 12: Download all outputs ─────────────────────────
import shutil
import os
from google.colab import files

interview_id = stage1_result.get('interview_id', 'interview')

# Ensure we're in the repository folder where outputs should exist
print('Current working dir:', os.getcwd())

# Check if outputs folder exists
if not os.path.exists('outputs'):
    print('✗ Error: outputs/ folder not found!')
    print('   Make sure you ran Cells 6-11 (all pipeline stages) first.')
    print('\nDirectory listing:')
    os.system('ls -la')
else:
    # List what's in outputs
    print('✔ Outputs folder found. Contents:')
    os.system('ls -lh outputs/ || true')

    # Zip the entire outputs folder
    zip_base = f'{interview_id}_outputs'
    zip_file = f'{zip_base}.zip'
    try:
        shutil.make_archive(
            base_name=zip_base,
            format='zip',
            root_dir='.',
            base_dir='outputs'
        )
    except Exception as e:
        print('✗ Error creating ZIP:', e)
    else:
        if os.path.exists(zip_file):
            size_mb = os.path.getsize(zip_file) / 1e6
            print(f'\n✔ Zipped outputs: {zip_file} ({size_mb:.1f} MB)')
            print('  Downloading...')
            files.download(zip_file)
        else:
            print(f'✗ Error: Could not create {zip_file}')


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import config
config.WHISPER_DEVICE = 'cpu'
config.WHISPER_COMPUTE_TYPE = 'int8'
config.WHISPER_LANGUAGE = 'ur'

from main import run_pipeline

run_pipeline('audio/test_audio.mp3', start_stage=1)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  URDU INTERVIEW PROCESSING PIPELINE
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  Audio file  : audio/test_audio.mp3
  Start stage : 1
  Started at  : 2026-07-01T18:18:34.561194

  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...
  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes for long audio)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.88 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.460 | conf=0.95 | and taking out the time...
    [⚠] 00:00:07.840 → 00:00:10.820 | conf=0.55 | If you give us a small introduction...
    [⚠] 00:00:1

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  ✔ Model loaded on cuda.

  Translating full text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  Translating segments:
    ✔ seg 001/537 | UR: اسلام علیکم Dr. Majda...
             | EN: Islam علیکم Dr. Majda...
    ✔ seg 002/537 | UR: Welcome to the show...
             | EN: Welcome to the show...
    ✔ seg 003/537 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today...
    ✔ seg 004/537 | UR: and taking out the time...
             | EN: today and taking the time...
    ✔ seg 005/537 | UR: If you give us a small introduction...
             | EN: if you give us a little introduction...
    ✔ seg 006/537 | UR: so that our audience that you are listen...
             | EN: so that our audience that you are listening to today...
    ✔ seg 007/537 | UR: they know that what a brilliant...
             | EN: today they know what a brilliant...
    ✔ seg 008/537 | UR: Academian you are...
             | EN: academic you are...
    ✔ seg 009/537 | UR: Thank you so much Zainab...
             | EN: Thank you so much Zainab...
    ✔ seg 010

    ✔ seg 373/537 | UR: your...
             | EN: your...
    ✔ seg 374/537 | UR: will...
             | EN: will...
    ✔ seg 375/537 | UR: is...
             | EN: is...
    ✔ seg 376/537 | UR: very important...
             | EN: very important...
    ✔ seg 377/537 | UR: and...
             | EN: and...


: 

In [16]:
# ── CELL 14: Evaluate pipeline against Ground Truth ─────────
# Compare Urdu transcription and English translation against ground truth files.

import os

if os.path.exists("evaluate.py"):
    print("Running evaluation...")
    !python evaluate.py \
      --urdu-hypothesis outputs/2_verified_transcripts/test_audio_verified_transcript.json \
      --english-hypothesis outputs/4_verified_translations/test_audio_verified_translation.json \
      --max-minutes 5
else:
    print("evaluate.py not found. Please ensure it is present in the project root.")

Running evaluation...


Error: Could not align hypothesis to ground truth. Anchor match 2/10 (20%) is below threshold 35%. Anchor tried: السلام علیکم بیٹے کے ساتھ ایسا...
